# Sleeper v3 — Interactive Chart Explorer

**This notebook is a thin wrapper.** All logic lives in `sleeper_core.py` (classes, chart methods)
and `data_loader.py` (disk-cached API loading). Nothing analytical should be written *in* this
notebook — if a chart needs changing, change `sleeper_core.py` and add a test in `tests/`.

The old 180-cell `Sleeper_v2.ipynb` is retired and lives at `archive/Sleeper_v2.ipynb`.
It is **stale** — `sleeper_core.py` moved past it (Playoffs, AllTimePlayoffs, Survivor,
PlayoffCalculator, plus many bug fixes exist only in the core). Its one unported feature is
**PickEm** (cells 34, 49–52, 100–102, 141) — kept as the reference for the planned webapp
Pick 'Em page (Session 11 in `design/review/synthesis.md`).

## The core classes

| Class | Constructor | Role |
|---|---|---|
| `League` | `League(year, league_id)` | settings, users, rosters, NFL schedule/stats |
| `Week` | `Week(week_num, league)` | one week's matchups + player breakouts |
| `Season` | `Season(league)` then `.Update()` | season aggregates; 30+ chart methods |
| `Playoffs` | `Playoffs(league, season)` | bracket resolution + playoff analytics |
| `AllTime` / `AllTimePlayoffs` | `AllTime()` / `AllTimePlayoffs()` | cross-season history (2019–present) |
| `SideBet` | `SideBet(league, season, weeks_dict)` | weekly side-bet charts |
| `Survivor` | `Survivor(year)` | survivor-pool picks & eliminations |
| `PlayoffCalculator` | `PlayoffCalculator(league, season, as_of_week)` | playoff-odds enumeration |

You rarely construct these by hand — `data_loader.load_data_for_year(year)` builds and caches
League + Weeks + Season for you (config comes from `config/*.json`).

In [ ]:
import data_loader as dl
import sleeper_core as core

YEAR = 2025
league, season, weeks = dl.load_data_for_year(YEAR)
print(f"{YEAR}: {len(weeks)} weeks loaded")

## Weekly charts — pick a `Week` object and call its methods

In [ ]:
wk = weeks[max(weeks)]          # latest loaded week (or weeks[5] etc.)
wk.WeeklyGraph().show()

In [ ]:
wk.PointsOverTheWeekend().show()

## Season charts

In [ ]:
season.PowerRankings(max(weeks)).show()

In [ ]:
season.LuckChart(max(weeks)).show()
# Other favorites:
# season.SeasonPointsForAgainst().show()
# season.ScoreTrends().show()
# season.ViolinPlayer(max(weeks), Starters=True).show()
# season.LineupEfficiencyChart(max(weeks)).show()

## Playoffs & All-Time

All-time classes read the module-global dicts, so load every year first.

In [ ]:
for y in core.AVAILABLE_YEARS:
    dl.load_data_for_year(y, verbose=False)

alltime = core.AllTime()
alltime.Update()
alltime.HallofFame_Team().show()

In [ ]:
playoffs = core.Playoffs(league, season)
# playoffs.ChampionRoad().show()
# playoffs.PlayoffHeatCheck().show()

# atp = core.AllTimePlayoffs()
# atp.PlayoffPedigree().show()

## Side bets & survivor

In [ ]:
sb = core.SideBet(league, season, weeks)
# Week methods exist per configured bet and take the Week object, e.g.:
# sb.Week5(weeks[5]).show()

# surv = core.Survivor(YEAR)

## Cache management

Season data is pickled in `.cache/`. To refresh a week after games finish:

In [ ]:
# dl.invalidate_week(YEAR, max(weeks))   # bust one week, then reload the year
# dl.clear_cache()                        # nuclear option: refetch everything